# 04 — Rhythm Features (Colab + Drive)

Writes `features/rhythm/rhythm_song.csv` from **AcousticBrainz / Essentia** JSON (not a mel-proxy). GPU Off. Needs 00+01.

Downloads `raw_30s_acousticbrainz-00..09` (same shard range as notebook 00) into `dataset/acousticbrainz/` if JSON files are not already on Drive.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
MEL_CACHE = Path("/content/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


def load_mel_npy(mel_abs, retries=5, pause=2.0):
    """Load mel from Drive with retries; cache on Colab disk to avoid FUSE drops."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-download its shard in notebook 00: {mel_abs} "
        f"({nbytes} bytes on Drive). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Download AcousticBrainz JSON (if missing) + extract rhythm fields


In [ ]:
import subprocess
from tqdm.auto import tqdm

if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)

AB_DIR = ROOT / "dataset" / "acousticbrainz"
AB_DIR.mkdir(parents=True, exist_ok=True)
AB_SHARDS = list(range(10))  # same 00–09 subset as notebook 00 mels
AB_URL = "https://cdn.freesound.org/mtg-jamendo/raw_30s/acousticbrainz"


def download_ab_shards():
    n_json = len(list(AB_DIR.rglob("*.json")))
    if n_json > 0:
        print(f"AcousticBrainz already on Drive: {n_json} JSON under {AB_DIR}")
        return
    if not check_internet("cdn.freesound.org") and not check_internet():
        raise RuntimeError(
            "No AcousticBrainz JSON on Drive and no Internet. "
            "Enable Internet, or run the official MTG script:\n"
            "  python3 scripts/download/download.py --dataset raw_30s "
            "--type acousticbrainz --from mtg-fast --unpack --remove "
            f"{AB_DIR}"
        )
    print("Downloading AcousticBrainz shards 00–09 to", AB_DIR)
    for i in AB_SHARDS:
        marker = AB_DIR / f".ab_shard_{i:02d}_done"
        if marker.exists():
            print(f"AB shard {i:02d} already done — skip")
            continue
        tar_name = f"raw_30s_acousticbrainz-{i:02d}.tar.gz"
        tar_path = AB_DIR / tar_name
        url = f"{AB_URL}/{tar_name}"
        print("Downloading", url)
        subprocess.check_call(["wget", "-q", "-O", str(tar_path), url])
        subprocess.check_call(["tar", "-xzf", str(tar_path), "-C", str(AB_DIR)])
        tar_path.unlink(missing_ok=True)
        marker.write_text("ok")
        print(f"AB shard {i:02d} saved")


def index_ab_json(root: Path) -> dict[str, Path]:
    idx = {}
    for p in root.rglob("*.json"):
        sid = normalize_track_id(p.stem)
        if sid:
            idx[sid] = p
    return idx


def _scalar(v):
    """Unwrap AcousticBrainz scalar or {mean: ...} stats. Never invent a default."""
    if v is None:
        return None
    if isinstance(v, dict):
        if "mean" in v:
            return _scalar(v["mean"])
        return None
    if isinstance(v, (list, tuple)):
        return None
    try:
        x = float(v)
    except (TypeError, ValueError):
        return None
    if np.isnan(x):
        return None
    return x


def rhythm_from_ab(doc: dict) -> dict | None:
    block = doc.get("rhythm")
    if not isinstance(block, dict):
        return None
    bpm = _scalar(block.get("bpm"))
    if bpm is None:
        return None
    beats_pos = block.get("beats_position")
    if not isinstance(beats_pos, (list, tuple)):
        beats_pos = []
    beats_count = _scalar(block.get("beats_count"))
    if beats_count is None and beats_pos:
        beats_count = float(len(beats_pos))
    intervals = np.diff(np.asarray(beats_pos, dtype=np.float64)) if len(beats_pos) > 1 else None
    feat = {
        "bpm": bpm,
        "beats_count": beats_count,
        "beats_loudness_mean": _scalar(block.get("beats_loudness")),
        "bpm_histogram_first_peak_bpm": _scalar(block.get("bpm_histogram_first_peak_bpm")),
        "bpm_histogram_first_peak_spread": _scalar(block.get("bpm_histogram_first_peak_spread")),
        "bpm_histogram_first_peak_weight": _scalar(block.get("bpm_histogram_first_peak_weight")),
        "onset_rate": _scalar(block.get("onset_rate")),
        "danceability": _scalar(block.get("danceability")),
        "beat_interval_mean": float(np.mean(intervals)) if intervals is not None else None,
        "beat_interval_std": float(np.std(intervals)) if intervals is not None else None,
    }
    return feat


download_ab_shards()
ab_index = index_ab_json(AB_DIR)
print("AcousticBrainz JSON indexed:", len(ab_index))

rows, missing = [], []
for _, rec in tqdm(manifest.iterrows(), total=len(manifest), desc="rhythm"):
    sid = str(rec["song_id"])
    path = ab_index.get(sid)
    if path is None:
        missing.append({"song_id": sid, "reason": "no_acousticbrainz_json"})
        continue
    try:
        doc = json.loads(path.read_text(encoding="utf-8", errors="replace"))
        feat = rhythm_from_ab(doc)
        if feat is None:
            missing.append({"song_id": sid, "reason": "json_missing_rhythm.bpm", "path": str(path)})
            continue
        feat.update({"song_id": sid, "source": "acousticbrainz", "split": rec["split"]})
        rows.append(feat)
    except Exception as e:
        missing.append({"song_id": sid, "reason": str(e), "path": str(path)})

n_mels = int(len(manifest))
n_ab_disk = int(len(ab_index))
n_written = int(len(rows))
n_excluded = int(len(missing))
print(f"songs with mels (manifest):     {n_mels}")
print(f"AcousticBrainz JSON on disk:    {n_ab_disk}")
print(f"overlap written to CSV:         {n_written}")
print(f"excluded (no/invalid AB JSON):  {n_excluded}")
if n_written == 0:
    raise RuntimeError("No overlapping AcousticBrainz rhythm rows — download AB shards and re-run.")

df = pd.DataFrame(rows)
out = FEAT_DIR / "rhythm"
out.mkdir(parents=True, exist_ok=True)
csv_path = out / "rhythm_song.csv"
df.to_csv(csv_path, index=False)
(RESULTS_DIR / "04_missing_acousticbrainz.json").write_text(json.dumps(missing, indent=2))
summary = {
    "n_manifest_mels": n_mels,
    "n_acousticbrainz_json": n_ab_disk,
    "n_overlap_written": n_written,
    "n_excluded": n_excluded,
    "csv": str(csv_path),
    "source": "acousticbrainz",
}
(RESULTS_DIR / "04_rhythm_summary.json").write_text(json.dumps(summary, indent=2))
print("wrote", csv_path, "rows=", n_written)
print(json.dumps(summary, indent=2))
df.head()
